<a href="https://colab.research.google.com/github/JustinZ-17/bytetrack-reproduction/blob/main/bytetrack_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Sun Sep  6 09:25:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
!git clone https://github.com/ifzhang/ByteTrack.git # 克隆仓库 对全会话有效
%cd ByteTrack

Cloning into 'ByteTrack'...
remote: Enumerating objects: 2007, done.
remote: Counting objects: 100% (329/329), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 2007 (delta 295), reused 287 (delta 287), pack-reused 1678 (from 1)
Receiving objects: 100% (2007/2007), 79.60 MiB | 25.83 MiB/s, done.
Resolving deltas: 100% (1159/1159), done.
/content/ByteTrack/ByteTrack


In [6]:
!pip3 install -r requirements.txt -q  #基础依赖安装
!python3 setup.py develop         #注册YOLOX进python
!pip3 install cython -q
!pip3 install 'git+https://github.com/cocodataset/cocoapi.git#subdirectory=PythonAPI' -q
!pip3 install cython_bbox -q        #C加速的IoU计算

  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Could not find a version that satisfies the requirement onnxruntime==1.8.0 (from versions: 1.20.0, 1.20.1, 1.21.0, 1.21.1, 1.22.0, 1.22.1, 1.23.0, 1.23.1, 1.23.2, 1.24.1, 1.24.2, 1.24.3, 1.24.4, 1.25.0, 1.25.1, 1.26.0, 1.27.0, 1.28.0, 1.29.0)
ERROR: No matching distribution found for onnxruntime==1.8.0
running develop
/usr/local/lib/python3.13/dist-packages/setuptools/command/develop.py:41: EasyInstallDeprecationWarning: easy_install command is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` and ``easy_install``.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://github.com/pypa/setuptools/issues/917 for detai

In [7]:
# 找到 yolox/ 目录下所有用 np.float 的文件，替换为 float
!grep -rl "np\.float\b" yolox/ | xargs sed -i 's/np\.float\b/float/g'

In [9]:
!mkdir -p pretrained    #下载与训练权重
!gdown --fuzzy 'https://drive.google.com/file/d/1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5/view?usp=sharing' -O pretrained/bytetrack_x_mot17.pth.tar

Downloading...
From (original): https://drive.google.com/uc?id=1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5
From (redirected): https://drive.google.com/uc?id=1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5&confirm=t&uuid=4e7b0268-001a-41e7-811d-dba0ad398ed7
To: /content/ByteTrack/ByteTrack/pretrained/bytetrack_x_mot17.pth.tar
100% 793M/793M [00:09<00:00, 81.5MB/s]


In [10]:
!ls -lh pretrained/ #确认一下权重文件

total 757M
-rw-r--r-- 1 root root 757M Sep 24  2021 bytetrack_x_mot17.pth.tar


In [20]:
!mkdir -p videos
# 下载测试视频
!wget -q https://github.com/opencv/opencv/raw/4.x/samples/data/vtest.avi -O videos/pedestrians.mp4
!ls -lh videos/

total 14M
-rw-r--r-- 1 root root 5.7M Sep  6 09:29 palace.mp4
-rw-r--r-- 1 root root 7.8M Sep  6 09:49 pedestrians.mp4


In [12]:
!pip install -q loguru lap easydict motmetrics tqdm pyyaml scipy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 18.1 MB/s eta 0:00:00


In [13]:
!pip install -q thop


In [14]:
!sed -i 's/ch = cv2.waitKey(1)/ch = -1/; s/ch = cv2.waitKey(0)/ch = -1/' tools/demo_track.py


In [15]:
!python3 tools/demo_track.py video \
    -f exps/example/mot/yolox_x_mix_det.py \
    -c pretrained/bytetrack_x_mot17.pth.tar \
    --fp16 --fuse --save_result \
    --path videos/pedestrians.mp4

2026-09-06 09:38:17.888 | INFO     | __main__:main:316 - Args: Namespace(demo='video', experiment_name='yolox_x_mix_det', name=None, path='videos/pedestrians.mp4', camid=0, save_result=True, exp_file='exps/example/mot/yolox_x_mix_det.py', ckpt='pretrained/bytetrack_x_mot17.pth.tar', device=device(type='cuda'), conf=None, nms=None, tsize=None, fps=30, fp16=True, fuse=True, trt=False, track_thresh=0.5, track_buffer=30, match_thresh=0.8, aspect_ratio_thresh=1.6, min_box_area=10, mot20=False)
/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
2026-09-06 09:38:20.217 | INFO     | __main__:main:326 - Model Summary: Params: 99.00M, Gflops: 793.21
2026-09-06 09:38:20.220 | INFO     | __main__:main:334 - loading checkpoint
2026-09-0

In [23]:
from google.colab import files
files.download("/YOLOX_outputs/yolox_x_mix_det/track_vis/2026_09_06_09_38_21/pedestrians.mp4")


FileNotFoundError: Cannot find file: /YOLOX_outputs/yolox_x_mix_det/track_vis/2026_09_06_09_38_21/pedestrians.mp4

In [18]:
!ls /content/
!ls /content/ByteTrack/YOLOX_outputs/yolox_x_mix_det/track_vis/ 2>/dev/null || echo "输出目录不存在"


ByteTrack  sample_data
输出目录不存在


In [24]:
!ls -la /content/ByteTrack/YOLOX_outputs/yolox_x_mix_det/track_vis/


ls: cannot access '/content/ByteTrack/YOLOX_outputs/yolox_x_mix_det/track_vis/': No such file or directory


In [25]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!git clone https://github.com/ifzhang/ByteTrack.git
%cd ByteTrack
!pip install -q loguru lap easydict motmetrics tqdm pyyaml scipy thop onnxruntime
!python setup.py develop
!sed -i 's/ch = cv2.waitKey(1)/ch = -1/; s/ch = cv2.waitKey(0)/ch = -1/' tools/demo_track.py
!mkdir -p pretrained


Mounted at /content/drive
/content
fatal: destination path 'ByteTrack' already exists and is not an empty directory.
/content/ByteTrack
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 78.1 MB/s eta 0:00:00
running develop
/usr/local/lib/python3.13/dist-packages/setuptools/command/develop.py:41: EasyInstallDeprecationWarning: easy_install command is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` and ``easy_install``.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://github.com/pypa/setuptools/issues/917 for details.
        ********************************************************************************

!!
  easy_install.initialize_options(self)
/usr/local/lib/python3.13/dist-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        *******************************

In [26]:
!gdown --fuzzy "https://drive.google.com/file/d/1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5/view?usp=sharing" -O pretrained/bytetrack_x_mot17.pth.tar


Downloading...
From (original): https://drive.google.com/uc?id=1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5
From (redirected): https://drive.google.com/uc?id=1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5&confirm=t&uuid=aa6cb407-6897-44e6-aa82-744405806861
To: /content/ByteTrack/pretrained/bytetrack_x_mot17.pth.tar
100% 793M/793M [00:16<00:00, 48.1MB/s]


In [27]:
!ls -lh pretrained/


total 757M
-rw-r--r-- 1 root root 757M Sep 24  2021 bytetrack_x_mot17.pth.tar


In [29]:
!ls pretrained/


bytetrack_x_mot17.pth.tar


In [30]:
!python3 tools/demo_track.py video \
    -f exps/example/mot/yolox_x_mix_det.py \
    -c pretrained/bytetrack_x_mot17.pth.tar \
    --fp16 --fuse --save_result \
    --path videos/pedestrians.mp4

2026-09-06 09:59:50.149 | INFO     | __main__:main:316 - Args: Namespace(demo='video', experiment_name='yolox_x_mix_det', name=None, path='videos/pedestrians.mp4', camid=0, save_result=True, exp_file='exps/example/mot/yolox_x_mix_det.py', ckpt='pretrained/bytetrack_x_mot17.pth.tar', device=device(type='cuda'), conf=None, nms=None, tsize=None, fps=30, fp16=True, fuse=True, trt=False, track_thresh=0.5, track_buffer=30, match_thresh=0.8, aspect_ratio_thresh=1.6, min_box_area=10, mot20=False)
/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
2026-09-06 09:59:52.612 | INFO     | __main__:main:326 - Model Summary: Params: 99.00M, Gflops: 793.21
2026-09-06 09:59:52.615 | INFO     | __main__:main:334 - loading checkpoint
2026-09-0

In [31]:
!mkdir -p /content/drive/MyDrive/ByteTrack_results
!cp -r YOLOX_outputs/yolox_x_mix_det/track_vis/* /content/drive/MyDrive/ByteTrack_results/baseline/


cp: target '/content/drive/MyDrive/ByteTrack_results/baseline/' is not a directory


In [32]:
!mkdir -p videos
!mv bytetrack_husty9-demo.mp4 videos/
!ls -lh videos/


mv: cannot stat 'bytetrack_husty9-demo.mp4': No such file or directory
total 5.7M
-rw-r--r-- 1 root root 5.7M Sep  6 09:25 palace.mp4


In [33]:
!python3 tools/demo_track.py video \
    -f exps/example/mot/yolox_x_mix_det.py \
    -c pretrained/bytetrack_x_mot17.pth.tar \
    --fp16 --fuse --save_result \
    --path videos/bytetrack_husty9-demo.mp4


2026-09-06 10:15:20.361 | INFO     | __main__:main:316 - Args: Namespace(demo='video', experiment_name='yolox_x_mix_det', name=None, path='videos/bytetrack_husty9-demo.mp4', camid=0, save_result=True, exp_file='exps/example/mot/yolox_x_mix_det.py', ckpt='pretrained/bytetrack_x_mot17.pth.tar', device=device(type='cuda'), conf=None, nms=None, tsize=None, fps=30, fp16=True, fuse=True, trt=False, track_thresh=0.5, track_buffer=30, match_thresh=0.8, aspect_ratio_thresh=1.6, min_box_area=10, mot20=False)
/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
2026-09-06 10:15:22.301 | INFO     | __main__:main:326 - Model Summary: Params: 99.00M, Gflops: 793.21
2026-09-06 10:15:22.305 | INFO     | __main__:main:334 - loading checkpoint

In [34]:
!mkdir -p /content/drive/MyDrive/ByteTrack_results/husty9
!cp -r YOLOX_outputs/yolox_x_mix_det/track_vis/* /content/drive/MyDrive/ByteTrack_results/husty9/


In [35]:
!find YOLOX_outputs -name "*.mp4" | xargs ls -lh


total 88K
drwxr-xr-x  2 root root 4.0K Sep  6 09:25 assets
drwxr-xr-x  4 root root 4.0K Sep  6 09:26 build
drwxr-xr-x 15 root root 4.0K Sep  6 09:38 ByteTrack
drwxr-xr-x  3 root root 4.0K Sep  6 09:25 datasets
drwxr-xr-x  6 root root 4.0K Sep  6 09:25 deploy
-rw-r--r--  1 root root 2.4K Sep  6 09:25 Dockerfile
drwxr-xr-x  4 root root 4.0K Sep  6 09:25 exps
-rw-r--r--  1 root root 1.1K Sep  6 09:25 LICENSE
drwxr-xr-x  2 root root 4.0K Sep  6 09:57 pretrained
-rw-r--r--  1 root root  16K Sep  6 09:25 README.md
-rw-r--r--  1 root root  260 Sep  6 09:25 requirements.txt
-rw-r--r--  1 root root  615 Sep  6 09:25 setup.cfg
-rw-r--r--  1 root root 1.7K Sep  6 09:25 setup.py
drwxr-xr-x  2 root root 4.0K Sep  6 09:55 tools
drwxr-xr-x 11 root root 4.0K Sep  6 09:25 tutorials
drwxr-xr-x  2 root root 4.0K Sep  6 09:25 videos
drwxr-xr-x 15 root root 4.0K Sep  6 09:59 yolox
drwxr-xr-x  2 root root 4.0K Sep  6 09:55 yolox.egg-info
drwxr-xr-x  3 root root 4.0K Sep  6 09:59 YOLOX_outputs


In [36]:
!ls YOLOX_outputs/yolox_x_mix_det/track_vis/
!find /content -name "bytetrack_husty9*" 2>/dev/null


2026_09_06_09_59_54	 2026_09_06_10_15_24
2026_09_06_09_59_54.txt  2026_09_06_10_15_24.txt
/content/bytetrack_husty9-demo.mp4


In [37]:
!mv /content/bytetrack_husty9-demo.mp4 videos/
!python3 tools/demo_track.py video \
    -f exps/example/mot/yolox_x_mix_det.py \
    -c pretrained/bytetrack_x_mot17.pth.tar \
    --fp16 --fuse --save_result \
    --path videos/bytetrack_husty9-demo.mp4


2026-09-06 10:18:54.026 | INFO     | __main__:main:316 - Args: Namespace(demo='video', experiment_name='yolox_x_mix_det', name=None, path='videos/bytetrack_husty9-demo.mp4', camid=0, save_result=True, exp_file='exps/example/mot/yolox_x_mix_det.py', ckpt='pretrained/bytetrack_x_mot17.pth.tar', device=device(type='cuda'), conf=None, nms=None, tsize=None, fps=30, fp16=True, fuse=True, trt=False, track_thresh=0.5, track_buffer=30, match_thresh=0.8, aspect_ratio_thresh=1.6, min_box_area=10, mot20=False)
/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
2026-09-06 10:18:56.367 | INFO     | __main__:main:326 - Model Summary: Params: 99.00M, Gflops: 793.21
2026-09-06 10:18:56.371 | INFO     | __main__:main:334 - loading checkpoint

In [38]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/ByteTrack_results
!cp -r YOLOX_outputs/yolox_x_mix_det/track_vis/* /content/drive/MyDrive/ByteTrack_results/


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [39]:
import cv2
for p in ["/content/bytetrack_husty9-demo.mp4",
          "YOLOX_outputs/yolox_x_mix_det/track_vis/2026_09_06_10_18_58/bytetrack_husty9-demo.mp4"]:
    cap = cv2.VideoCapture(p)
    print(p, "->", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), "frames")
    cap.release()


/content/bytetrack_husty9-demo.mp4 -> -1 frames
YOLOX_outputs/yolox_x_mix_det/track_vis/2026_09_06_10_18_58/bytetrack_husty9-demo.mp4 -> 508 frames


In [40]:
!ls /content/ByteTrack && ls /content/bytetrack_husty9-demo.mp4


assets	   deploy      pretrained	 setup.py   yolox
build	   Dockerfile  README.md	 tools	    yolox.egg-info
ByteTrack  exps        requirements.txt  tutorials  YOLOX_outputs
datasets   LICENSE     setup.cfg	 videos
ls: cannot access '/content/bytetrack_husty9-demo.mp4': No such file or directory


In [41]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r YOLOX_outputs/yolox_x_mix_det/track_vis/ /content/drive/MyDrive/ByteTrack_results/


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [43]:
!mkdir -p /content/drive/MyDrive/ByteTrack_results
!cp -r YOLOX_outputs/yolox_x_mix_det/track_vis/* /content/drive/MyDrive/ByteTrack_results/


In [44]:
!ls -R /content/drive/MyDrive/ByteTrack_results/ | head -30


/content/drive/MyDrive/ByteTrack_results/:
2026_09_06_09_59_54
2026_09_06_09_59_54.txt
2026_09_06_10_15_24
2026_09_06_10_15_24.txt
2026_09_06_10_18_58
2026_09_06_10_18_58.txt
husty9
track_vis

/content/drive/MyDrive/ByteTrack_results/2026_09_06_09_59_54:

/content/drive/MyDrive/ByteTrack_results/2026_09_06_10_15_24:

/content/drive/MyDrive/ByteTrack_results/2026_09_06_10_18_58:
bytetrack_husty9-demo.mp4

/content/drive/MyDrive/ByteTrack_results/husty9:
2026_09_06_09_59_54
2026_09_06_09_59_54.txt
2026_09_06_10_15_24
2026_09_06_10_15_24.txt

/content/drive/MyDrive/ByteTrack_results/husty9/2026_09_06_09_59_54:

/content/drive/MyDrive/ByteTrack_results/husty9/2026_09_06_10_15_24:

/content/drive/MyDrive/ByteTrack_results/track_vis:
2026_09_06_09_59_54
2026_09_06_09_59_54.txt
